## Cell 1 — Imports and Dataset configuration

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from skimage.transform import resize
from skimage import exposure

SEED = 42
np.random.seed(SEED)

# Find repository root from the notebook's working directory
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "Datasets" / "Brain Tumor 12K Mri Images" / "DATA.json").exists()
)

DATA_ROOT = PROJECT_ROOT / "Datasets" / "Brain Tumor 12K Mri Images"
IMAGE_ROOT = DATA_ROOT / "Images_" / "Images_"

PREPROCESS_DIR = PROJECT_ROOT / "Code" / "preprocessing"
MANIFEST_DIR = PREPROCESS_DIR / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (224, 224)

print("Project root:", PROJECT_ROOT)
print("Dataset found:", DATA_ROOT.exists())
print("Image root found:", IMAGE_ROOT.exists())

## Cell 2 — Generate the master dataset manifest

In [ ]:
with open(DATA_ROOT / "DATA.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

records = []

for relative_path, info in metadata.items():

    image_path = IMAGE_ROOT / Path(relative_path)

    if not image_path.is_file():
        continue

    label = info.get("has_lesion")

    if label not in (0, 1, False, True):
        continue

    records.append({
        "image_path": str(image_path.resolve()),
        "label": int(label),
        "tumor_type": info.get("tumor_type"),
        "sequence": info.get("sequence"),
        "patient_id": info.get("patient_id"),
        "mask_path": info.get("mask_path"),
        "bbox_path": info.get("bbox_path"),
        "original_width": info.get("width"),
        "original_height": info.get("height")
    })

df = pd.DataFrame(records)

assert not df.empty, "No valid MRI images found."
assert df["image_path"].is_unique, "Duplicate image paths detected."

MANIFEST_PATH = MANIFEST_DIR / "brain_mri_master_manifest.csv"
df.to_csv(MANIFEST_PATH, index=False)

print("Total MRI images:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts())

print("\nMRI sequences:")
print(df["sequence"].value_counts())

print("\nSaved manifest:", MANIFEST_PATH)

## Cell 3 — Define standardized MRI preprocessing

In [ ]:
def preprocess_mri(
    image_path,
    image_size=(224, 224),
    use_clahe=False
):
    """
    Standardized MRI preprocessing.

    Steps:
    1. Load image
    2. Convert to grayscale
    3. Resize
    4. Robust intensity normalization
    5. Optional CLAHE

    Returns:
        Float32 array with values in [0, 1].
    """

    # Load grayscale MRI
    with Image.open(image_path) as img:
        image = np.asarray(
            img.convert("L"),
            dtype=np.float32
        )

    # Resize to a consistent spatial resolution
    image = resize(
        image,
        image_size,
        preserve_range=True,
        anti_aliasing=True
    ).astype(np.float32)

    # Robust percentile normalization
    p1, p99 = np.percentile(image, [1, 99])

    if p99 > p1:
        image = np.clip(image, p1, p99)
        image = (image - p1) / (p99 - p1)
    else:
        image = np.zeros_like(image)

    # Optional contrast enhancement
    if use_clahe:
        image = exposure.equalize_adapthist(
            image,
            clip_limit=0.01
        )

    return image.astype(np.float32)

## Cell 4 — Visualize the preprocessing results

In [ ]:
sample_path = df.iloc[0]["image_path"]

with Image.open(sample_path) as img:
    original = np.asarray(img.convert("L"))

normalized = preprocess_mri(
    sample_path,
    image_size=IMAGE_SIZE,
    use_clahe=False
)

clahe = preprocess_mri(
    sample_path,
    image_size=IMAGE_SIZE,
    use_clahe=True
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original, cmap="gray")
axes[0].set_title("Original MRI")

axes[1].imshow(normalized, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Normalized MRI")

axes[2].imshow(clahe, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("MRI with CLAHE")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Original shape:", original.shape)
print("Preprocessed shape:", normalized.shape)
print("Intensity range:", normalized.min(), normalized.max())